# Pelagic spin-up validation notebook

This notebook follows the following workflow:

1. **Configuration** for paths, variables, layers, and output folders.
2. **Helper functions** for loading model output and handling grid/time quirks.
3. **Analysis functions** for trends, subdomain diagnostics, observations, and satellite comparison.
4. **Entry-point examples** that can be toggled on when you want to run a specific workflow.

The main cleanup change is that map-based diagnostics now **export individual PNG frames** instead of rendering animations inline in the notebook.


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from IPython.display import display
import os


In [ ]:
# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------

POSTPROC_DIR = Path('/export/lv9/projects/dws/results/validation/pelagic/')
BASE_OUTPUT_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/')

base_dir = "/export/lv9/projects/dws/model_output/archived_runs"
frames_root = "/export/lv9/projects/dws/results/animation/pelagic/frames_all_spinups"
os.makedirs(frames_root, exist_ok=True)

DEFAULT_SPINUP = 'spinup_05'
YEAR = 2015

FRAME_OUTPUT_DIR = (
    POSTPROC_DIR / DEFAULT_SPINUP / f"{YEAR}_spinup_frames_basemap" / variable
)
FRAME_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Map window
MAP_EXTENT_LONLAT = (4.0, 6.6, 52.5, 53.8)

# Transect definition
I_LON = 152
LAT_START = 90
LAT_STOP = 153

# Color scale
VMIN = 0
VMAX = 4000
CMAP = cmocean.cm.matter


# Choose ONE variable
variable = "P1c"
label = "Diatoms (P1c)"

# variable = "P2c"
# label = "Flagellates (P2c)"

# variable = "P3c"
# label = "PicoPhytoPlankton (P3c)"



In [ ]:


spinups = sorted([d for d in os.listdir(base_dir) if d.startswith("spinup_")])
frame_counter = 0

for spinup in spinups:
    print(f"\n=== Processing spinup: {spinup} ===")

    spinup_dir = os.path.join(base_dir, spinup)
    nc_files = sorted(glob.glob(os.path.join(spinup_dir, "dws_500m.3d.*.nc")))

    # Create subfolder for this spinup
    spinup_frame_dir = os.path.join(frames_root, spinup)
    os.makedirs(spinup_frame_dir, exist_ok=True)

    for file_path in nc_files:
        print(f"  → Reading {file_path}")

        with xr.open_dataset(file_path) as ds:

            # Fix curvilinear grid
            lon_da = ds.lonc.ffill("xc").bfill("xc").ffill("yc").bfill("yc")
            lat_da = ds.latc.ffill("xc").bfill("xc").ffill("yc").bfill("yc")
            lon = lon_da.values
            lat = lat_da.values

            # Extract variable
            data_var = ds[variable]
            if "level" in data_var.dims:
                data_var = data_var.isel(level=-1)

            # Mask variable
            mask_da = ds["elev"].squeeze(drop=True)
            mask_time_dim = _find_time_dim(mask_da)
            mask_da = _drop_duplicate_time(mask_da, mask_time_dim)

            # Build transect
            tran_lon = lon[LAT_START:LAT_STOP, I_LON]
            tran_lat = lat[LAT_START:LAT_STOP, I_LON]
            valid = np.isfinite(tran_lon) & np.isfinite(tran_lat)
            tran_lon = tran_lon[valid]
            tran_lat = tran_lat[valid]

            # Loop through time steps
            for t_idx in range(data_var.sizes["time"]):

                frame = data_var.isel(time=t_idx).values.astype(float)

                # Apply mask
                mask2d = mask_da.isel({mask_time_dim: t_idx}).values
                wet_mask = mask2d > -1.1869
                frame[~wet_mask] = np.nan

                timestamp = str(ds.time.values[t_idx])[:10]  # YYYY-MM-DD

                # Plot
                fig = plt.figure(figsize=(8, 6))
                ax = plt.axes(projection=ccrs.Mercator())
                ax.set_extent(MAP_EXTENT_LONLAT, crs=ccrs.PlateCarree())

                # Basemap
                ctx.add_basemap(
                    ax,
                    crs=ax.projection.to_string(),
                    source=ctx.providers.Esri.WorldShadedRelief,
                    zorder=1,
                )
                txt = ax.texts[-1]
                txt.set_position([0.99, 0.02])
                txt.set_ha("right")

                # Field
                mesh = ax.pcolormesh(
                    lon,
                    lat,
                    frame,
                    transform=ccrs.PlateCarree(),
                    cmap=CMAP,
                    vmin=VMIN,
                    vmax=VMAX,
                    shading="gouraud",
                    zorder=2,
                )

                # Transect
                ax.plot(
                    tran_lon,
                    tran_lat,
                    "--",
                    color="black",
                    linewidth=2,
                    transform=ccrs.PlateCarree(),
                    zorder=5,
                )

                # Colorbar
                cbar = plt.colorbar(mesh, ax=ax, shrink=0.9, pad=0.05)
                cbar.set_label("Biomass (mgC m$^{-3}$)", fontsize=12)

                ax.set_title(f"{label} | {spinup} | {timestamp}")

                # Save frame with:
                #   - global counter (for MP4)
                #   - date (for readability)
                frame_path = os.path.join(
                    spinup_frame_dir,
                    f"frame_{frame_counter:06d}_{timestamp}.png"
                )

                plt.savefig(frame_path, dpi=200, bbox_inches="tight")
                plt.close(fig)

                frame_counter += 1


In [ ]:
def export_mp4_from_frames(frame_dir,
                           output_name="animation.mp4",
                           fps=40): 

    frame_dir = Path(frame_dir)
    output_path = Path(output_name)

    frame_files = sorted(frame_dir.rglob("*.png"))
    if not frame_files:
        print(f"No PNG frames found in {frame_dir}")
        return

    print(f"Creating MP4 animation → {output_path}")
    print(f"Found {len(frame_files)} frames")

    writer = animation.FFMpegWriter(
        fps=fps,
        codec="libx264",
        bitrate=1800,
        extra_args=["-pix_fmt", "yuv420p"]   # <-- Windows compatible
    )

    first_img = plt.imread(frame_files[0])
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(first_img)
    ax.axis("off")

    with writer.saving(fig, str(output_path), dpi=200):
        for f in frame_files:
            img = plt.imread(f)
            im.set_data(img)
            writer.grab_frame()

    plt.close(fig)
    print(f"MP4 saved: {output_path}")


In [ ]:
export_mp4_from_frames(
    frames_root,
    output_name=Path(frames_root) / "pelagic_all_spinups.mp4",
    fps=25
)


In [ ]:
# -----------------------------------------------------------------------------
# HELPER
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# FRAME EXPORT WITHOUT BASEMAP
# -----------------------------------------------------------------------------

def export_variable_frames(ds: xr.Dataset) -> None:
    print(f"Exporting frames for {variable} …")

    export_map_frames(
        ds,
        plotvar=variable,
        output_dir=FRAME_OUTPUT_DIR,
        layer_index=MODEL_SURFACE_LAYER_INDEX,
        time_step=1,
        cmap=CMAP,
        vmin=VMIN,
        vmax=VMAX,
        mask_var=MASK_VAR,
        mask_threshold=MASK_THRESHOLD,
    )

    print(f"Frames saved to: {FRAME_OUTPUT_DIR}")

def export_map_frames(
    ds: xr.Dataset,
    *,
    plotvar: str,
    output_dir: Path,
    layer_index: int | None,
    time_step: int,
    cmap: str,
    vmin: float,
    vmax: float,
    mask_var: str | None = None,
    mask_threshold: float | None = None,
) -> None:
    _apply_theme()
    if plotvar not in ds.variables:
        raise KeyError(f"Variable '{plotvar}' not found. Available: {sorted(ds.data_vars)}")
    if vmin is None or vmax is None:
        raise ValueError('vmin and vmax must both be provided for consistent frame export.')

    da = ds[plotvar].squeeze(drop=True)
    time_dim = _find_time_dim(da)
    field, z_dim = _resolve_layer(da, layer_index, time_dim)
    field = _drop_duplicate_time(field, time_dim)
    y_dim, x_dim = _find_xy_dims(field, (time_dim,))
    x_plot, y_plot, use_geo = _build_horizontal_coords(ds, y_dim, x_dim)
    layer_text = '' if z_dim is None else f' | layer {layer_index}'

    if mask_var is not None:
        if mask_var not in ds.variables:
            raise KeyError(f"Mask variable '{mask_var}' not found.")
        mask_da = ds[mask_var].squeeze(drop=True)
        mask_time_dim = _find_time_dim(mask_da)
        mask_da = _drop_duplicate_time(mask_da, mask_time_dim)
        field, mask_da = xr.align(field, mask_da, join='inner')
    else:
        mask_da = None

    frame_indices = np.arange(0, field.sizes[time_dim], time_step, dtype=int)
    if frame_indices.size == 0:
        raise ValueError('No frames selected. Check time_step and time length.')

    output_dir.mkdir(parents=True, exist_ok=True)
    print(f'Exporting {frame_indices.size} frames to: {output_dir}')

    for frame_number, frame_idx in enumerate(frame_indices):
        frame2d = _to_float(field.isel({time_dim: int(frame_idx)}).transpose(y_dim, x_dim).values)
        if mask_da is not None:
            wet_mask = _to_float(mask_da.isel({time_dim: int(frame_idx)}).transpose(y_dim, x_dim).values) > mask_threshold
            frame2d[~wet_mask] = np.nan

        timestamp = pd.Timestamp(field[time_dim].values[int(frame_idx)])
        frame_name = f"frame_{frame_number:04d}_{_format_timestamp_for_filename(timestamp)}.png"
        frame_path = output_dir / frame_name

        fig, ax = plt.subplots(figsize=(8, 6))
        mesh = ax.pcolormesh(
            x_plot,
            y_plot,
            np.ma.masked_invalid(frame2d),
            shading='auto',
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
        )
        cbar = fig.colorbar(mesh, ax=ax, fraction=0.035, pad=0.03)
        units = da.attrs.get('units', '')
        cbar.set_label(f'{plotvar} [{units}]' if units else plotvar)
        ax.set_title(f'{plotvar}{layer_text} | {timestamp}')
        ax.set_xlabel('Longitude' if use_geo else x_dim)
        ax.set_ylabel('Latitude' if use_geo else y_dim)
        ax.grid(True, alpha=0.2)
        plt.tight_layout()
        fig.savefig(frame_path, dpi=200, bbox_inches='tight')
        plt.close(fig)

    print(f'Saved {frame_indices.size} frames: {output_dir}')

# -----------------------------------------------------------------------------
# FRAME EXPORT WITH BASEMAP + TRANSECT
# -----------------------------------------------------------------------------

def export_variable_frames_with_basemap(ds: xr.Dataset,
                                        *,
                                        plotvar: str,
                                        output_dir: Path,
                                        layer_index: int,
                                        time_step: int,
                                        cmap: str,
                                        vmin: float,
                                        vmax: float,
                                        mask_var: str | None = None,
                                        mask_threshold: float | None = None,
                                        ) -> None:

    print(f"Exporting frames with basemap + transect + mask for {plotvar} …")

    # -------------------------------------------------------------------------
    # Extract model variable
    # -------------------------------------------------------------------------
    da = ds[plotvar].squeeze(drop=True)
    time_dim = _find_time_dim(da)

    # Resolve vertical layer (same as original)
    field, z_dim = _resolve_layer(da, layer_index, time_dim)
    field = _drop_duplicate_time(field, time_dim)

    # Identify horizontal dims
    y_dim, x_dim = _find_xy_dims(field, (time_dim,))

    # Curvilinear lon/lat
    lon_plot, lat_plot, _ = _build_horizontal_coords(ds, y_dim, x_dim, fill_invalid=True)

    # -------------------------------------------------------------------------
    # MASK HANDLING — EXACTLY LIKE ORIGINAL FUNCTION
    # -------------------------------------------------------------------------
    if mask_var is not None:
        if mask_var not in ds.variables:
            raise KeyError(f"Mask variable '{mask_var}' not found.")

        mask_da = ds[mask_var].squeeze(drop=True)
        mask_time_dim = _find_time_dim(mask_da)
        mask_da = _drop_duplicate_time(mask_da, mask_time_dim)

        # Align mask and field
        field, mask_da = xr.align(field, mask_da, join='inner')
    else:
        mask_da = None

    # -------------------------------------------------------------------------
    # TRANSECT (curvilinear)
    # -------------------------------------------------------------------------
    tran_lon = lon_plot[LAT_START:LAT_STOP, I_LON]
    tran_lat = lat_plot[LAT_START:LAT_STOP, I_LON]
    valid = np.isfinite(tran_lon) & np.isfinite(tran_lat)
    tran_lon = tran_lon[valid]
    tran_lat = tran_lat[valid]

    # -------------------------------------------------------------------------
    # FRAME INDICES
    # -------------------------------------------------------------------------
    frame_indices = np.arange(0, field.sizes[time_dim], time_step, dtype=int)
    if frame_indices.size == 0:
        raise ValueError("No frames selected. Check time_step and time length.")

    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Saving {frame_indices.size} frames → {output_dir}")

    # -------------------------------------------------------------------------
    # LOOP THROUGH FRAMES
    # -------------------------------------------------------------------------
    for frame_number, frame_idx in enumerate(frame_indices):

        # Extract 2D field
        frame2d = _to_float(
            field.isel({time_dim: int(frame_idx)}).transpose(y_dim, x_dim).values
        )

        # Apply mask (same logic as original)
        if mask_da is not None:
            mask2d = _to_float(
                mask_da.isel({mask_time_dim: int(frame_idx)}).transpose(y_dim, x_dim).values
            )
            wet_mask = mask2d > mask_threshold
            frame2d[~wet_mask] = np.nan

        timestamp = pd.Timestamp(field[time_dim].values[int(frame_idx)])
        fname = f"frame_{frame_number:04d}_{_format_timestamp_for_filename(timestamp)}.png"
        fpath = output_dir / fname

        # ---------------------------------------------------------------------
        # PLOTTING WITH BASEMAP + TRANSECT
        # ---------------------------------------------------------------------
        fig = plt.figure(figsize=(8, 6))
        ax = plt.axes(projection=ccrs.Mercator())
        ax.set_extent(MAP_EXTENT_LONLAT, crs=ccrs.PlateCarree())

        # Basemap
        ctx.add_basemap(
            ax,
            crs=ax.projection.to_string(),
            source=ctx.providers.Esri.WorldShadedRelief,
            zorder=1,
        )
        txt = ax.texts[-1]
        txt.set_position([0.99, 0.02])
        txt.set_ha("right")

        # Model field
        mesh = ax.pcolormesh(
            lon_plot,
            lat_plot,
            np.ma.masked_invalid(frame2d),
            transform=ccrs.PlateCarree(),
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            shading="gouraud",
            zorder=2,
        )

        # Transect
        ax.plot(
            tran_lon,
            tran_lat,
            linestyle="--",
            color="black",
            linewidth=2,
            transform=ccrs.PlateCarree(),
            zorder=5,
        )

        # Colorbar
        cbar = plt.colorbar(mesh, ax=ax, shrink=0.9, pad=0.05)
        units = da.attrs.get("units", "")
        cbar.set_label(f"{plotvar} [{units}]" if units else plotvar)

        ax.set_title(f"{plotvar} | {timestamp.date()}", fontsize=12)

        fig.savefig(fpath, dpi=200, bbox_inches="tight")
        plt.close(fig)

    print(f"Done. Frames saved in: {output_dir}")




